In [27]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re


In [28]:
def load_json_records(path: str) -> list[dict]:
    p = Path(path)
    files = sorted(p.glob("**/*.json")) if p.is_dir() else [p]
    if not files:
        raise FileNotFoundError(f"No .json files found at: {path}")

    records = []
    for fp in files:
        raw = fp.read_text(encoding="utf-8").strip()
        try:
            data = json.loads(raw)
            if isinstance(data, list):
                records.extend(data)
            elif isinstance(data, dict):
                records.append(data)
        except json.JSONDecodeError:
            for line in raw.splitlines():
                line = line.strip().rstrip(",")
                if line:
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass
    return records

def safe_filename(name: str, max_len: int = 60) -> str:
    name = re.sub(r'[\\/:*?\"\'<>|,]', '', name)
    name = name.strip().replace(" ", "_")
    return name[:max_len]

def normalize_standard(val):
    try:
        return int(val)
    except (ValueError, TypeError):
        match = re.search(r'\d+', str(val))
        return int(match.group()) if match else val

In [29]:
LBA_QUESTIONS_PATH = "C:\\Users\\adity\\OneDrive\\Documents\\Sem 6\\Shiksha_Copilot_Logs\\Shiksha_Copilot_DB\\lba_questions.json"
CHAPTERS_PATH      = "C:\\Users\\adity\\OneDrive\\Documents\\Sem 6\\Shiksha_Copilot_Logs\\Shiksha_Copilot_DB\\chapters.json"
OUTPUT_DIR         = "C:\\Users\\adity\\OneDrive\\Documents\\Sem 6\\Shiksha_Copilot_Logs\\lba_sorted_data"

lba_questions = load_json_records(LBA_QUESTIONS_PATH)
chapters_raw  = load_json_records(CHAPTERS_PATH)

In [30]:
chapter_lookup = {}
for ch in chapters_raw:
    cid = (ch.get("_id") or {}).get("$oid", "")
    if cid:
        chapter_lookup[cid] = {
            "chapter_title":  ch.get("topics", "Unknown"),
            "medium":         ch.get("medium"),
        }

In [31]:
rows = []
for q in lba_questions:
    chapter_id = (q.get("chapterId") or {}).get("$oid", "")
    ch_info    = chapter_lookup.get(chapter_id, {})
    rows.append({
        "question_id":      (q.get("_id") or {}).get("$oid", ""),
        "subject":          q.get("subject", "unknown").lower(),
        "class":            q.get("class", ""),
        "chapter_number":   (q.get("chapter") or {}).get("chapterNumber"),
        "chapter_title":    (q.get("chapter") or {}).get("title") or ch_info.get("chapter_title", ""),
        "medium":           q.get("medium", ""),
        "difficulty":       q.get("difficulty", ""),
        "answer_type":      q.get("answerType", ""),
        "marks":            q.get("marksPerQuestion"),
        "text":             q.get("text", ""),
        "options":          json.dumps([o.get("text") for o in q.get("options", [])]),
        "key_answer":       q.get("keyAnswer", ""),
        "exam_type":        q.get("examType", ""),
        "year":             q.get("year", ""),
        "group_heading":    q.get("groupHeading", ""),
        "chapter_id":       chapter_id,
    })


In [32]:
df_questions = pd.DataFrame(rows)
df_questions['standard'] = df_questions['class'].apply(normalize_standard)

df_questions.sort_values(
    ["subject", "class", "chapter_number"],
    inplace=True,
    na_position="last"
)

missing_chapter = df_questions[df_questions["chapter_title"] == "Unknown"]
print(f"Questions with unresolved chapter_title: {len(missing_chapter):,} / {len(df_questions):,}")
print(f"Sample chapter_ids with no match:\n{missing_chapter['chapter_id'].value_counts().head(5)}")

print(f"\nStandard value counts:\n{df_questions['class'].value_counts().head(10)}")
print(f"Chapter title value counts:\n{df_questions['chapter_title'].unique()}")

Questions with unresolved chapter_title: 0 / 7,494
Sample chapter_ids with no match:
Series([], Name: count, dtype: int64)

Standard value counts:
class
9          3088
5          1397
7          1195
6           735
10          721
8           343
6th std      15
Name: count, dtype: int64
Chapter title value counts:
<StringArray>
[                                 'PROSE: A Hero',
                    'POEM: Grandma Climbs a Tree',
         'PROSE: "There's a Girl by the Tracks!"',
                         'POEM: Quality of Mercy',
               'PROSE: Gentleman of Rio en Medio',
                            'POEM: I Am the Land',
                       'PROSE: Dr. B.R. Ambedkar',
                        'POEM: The Song Of India',
                             'PROSE: The Concert',
                            'POEM: Jazz Poem Two',
 ...
             'FREEDOM MOVEMENT (1885 - 1919 C.E)',
   'UNIFICATION OF KARNATAKA AND BORDER DISPUTES',
              'PRO-SOCIAL MOVEMENTS OF KARNATAKA',

In [33]:
output_path = Path(OUTPUT_DIR)

df_questions['subject'] = df_questions['subject'].str.replace('_', ' ').str.lower().str.strip()

for subject, subj_group in df_questions.groupby("subject"):
    subj_dir = output_path / safe_filename(subject)
    
    for standard, std_group in subj_group.groupby("standard"):
        std_dir = subj_dir / f"Class_{standard}"
        std_dir.mkdir(parents=True, exist_ok=True)
        
        for chapter, ch_group in std_group.groupby("chapter_title"):
            out_file = std_dir / f"{safe_filename(chapter)}.csv"
            ch_group.to_csv(out_file, index=False)